# Welcome to PYNQ Audio
This notebook shows the basic recording and playback features of the AUP-ZU3.  
It uses the audio jack `HP+MIC` to play back recordings; it can take inputs from 
the microphone on `HP+MIC` or `LINE_IN`. Pre-recorded audio sample can also be taken
as input. Moreover, visualization with matplotlib is shown.
## Create new audio object

In [1]:
from pynq import Overlay, allocate
import time
from aic3104 import AIC3104
import numpy as np
from scipy.io import wavfile
import asyncio
import wave

# 1. Load the Overlay
ol = Overlay("hw_wrapper.bit")

# 2. Check IP Dictionary
print("Available IPs:", ol.ip_dict.keys())

# release audio reset
rst = ol.audio.axi_gpio_0.channel1 
rst.write(0x1, 0x1)

codec = AIC3104(ol.audio.axi_iic_0)
print("on the bus:", codec.scan())          # should now list 0x18
codec.reset_controller()            # clear a wedged AXI-IIC core if needed
print("on the bus:", codec.scan())          # should now list 0x18
codec.init(mclk_hz=12_288_000, fs=48000, word_len=16)     # set mclk_hz to YOUR design's MCLK
codec.select_microphone()
print(codec.dump())
dma = ol.audio.aic3104_dma_wrapper_0

Available IPs: dict_keys(['audio/axi_gpio_0', 'audio/axi_iic_0', 'audio/aic3104_dma_wrapper_0', 'zynq_ultra_ps_e_0'])
on the bus: ['0x18']
on the bus: ['0x18']
{'sample_rate(2)': '0x0', 'pll_a(3)': '0x10', 'datapath(7)': '0xa', 'serial_a(8)': '0x0', 'serial_b(9)': '0xf0', 'clkgen(101)': '0x0', 'clkdiv(102)': '0x2', 'dac_power(37)': '0xc0', 'ldac_vol(43)': '0x0', 'leftlop(86)': '0xb'}


In [2]:
# define our registers
REG_TX_ADDR_LO    = 0x0
REG_TX_ADDR_HI    = 0x4
REG_TX_BYTES      = 0x8
REG_TX_START      = 0xc
REG_TX_BYTES_READ = 0x10
REG_TX_STAT       = 0x14
REG_RX_ADDR_LO    = 0x100
REG_RX_ADDR_HI    = 0x104
REG_RX_BYTES      = 0x108
REG_RX_START      = 0x10c
REG_RX_BYTES_READ = 0x110
REG_RX_STAT       = 0x114
REG_DEV           = 0x200
REG_VER           = 0x204
REG_INT           = 0x208
DONE_BIT          = 1 << 31

# Read the i2s version
device = dma.read(REG_DEV)
version = dma.read(REG_VER)
print(f"I2S Device = {(device)}, Version = {(version)}")
print("Starting Capture")
# We want to sample 5 seconds of 48Khz audio
TIME = 5
SAMPLE_RATE = 48000  # set this to your I2S/codec rate (44100, 48000, etc.)
N = TIME * SAMPLE_RATE
buf = allocate(shape=(N,), dtype=np.uint32)   # default: non-cacheable -> HP port

dma.write(REG_RX_ADDR_LO, buf.device_address & 0xFFFFFFFF)
dma.write(REG_RX_ADDR_HI, buf.device_address >> 32)   # needed now that addr > 32 bits
dma.write(REG_RX_BYTES,   buf.nbytes)
dma.write(REG_RX_START,   1)                          # cfg_start pulse
while not (dma.read(REG_RX_STAT) & DONE_BIT):
    pass

print("Capture Complete!")
# --- Unpack the 32-bit word into two signed 16-bit samples ---
# Assumes left in the upper 16 bits, right in the lower 16 bits.
left  = (buf >> 16).astype(np.uint16).view(np.int16)
right = (buf & 0xFFFF).astype(np.uint16).view(np.int16)

# --- Interleave L R L R for a stereo WAV ---
stereo = np.empty(N * 2, dtype=np.int16)
stereo[0::2] = left
stereo[1::2] = right

# --- Write the WAV ---
with wave.open("capture.wav", "wb") as w:
    w.setnchannels(2)
    w.setsampwidth(2)          # 2 bytes = 16 bits
    w.setframerate(SAMPLE_RATE)
    w.writeframes(stereo.tobytes())   

I2S Device = 858861620, Version = 2
Starting Capture
Capture Complete!


## Play back the capture (DMA read / MM2S)
Stream the captured buffer back out through the codec using the `axi_dma_reader`
(MM2S) engine. The reader walks the DDR buffer and feeds the I2S TX FIFO, which
the DAC drains one packed L/R sample per 48&nbsp;kHz frame. Audio comes out of the
**LINE OUT** jack (green, J4).

The capture buffer `buf` is already in the exact 32-bit packing the TX path
expects (`bits[31:16]=left`, `bits[15:0]=right` &mdash; the same words the capture
engine wrote to DDR), so we replay it **verbatim**: no WAV round-trip and no L/R
re-packing. Drive the `REG_TX_*` registers exactly like capture used `REG_RX_*`,
then poll `REG_TX_STAT` for the done bit.

> **Note on the done bit.** `REG_TX_STAT[31]` (done) is *sticky*: it stays set
> from the previous playback until the next `START` clears it. So we wait for it
> to **drop** (the reader accepted the new transfer) and then **rise** again
> (this transfer finished). Polling only for "set" would fall through instantly
> on a re-run, reading back a stale completion.

In [ ]:
# --- Playback via axi_dma_read (MM2S) ---
# Replay the just-captured buffer back out through the codec. `buf` already holds
# the 32-bit packed L/R words the capture engine wrote to DDR, so we stream it
# back verbatim -- the reader (axi_dma_reader / MM2S) walks the buffer and feeds
# the I2S TX FIFO, and the DAC drives LINE OUT (green jack J4, routed by
# codec.init()). No WAV round-trip, no re-packing, so L/R can't get swapped.

UNDERFLOW_BIT = 1 << 3        # REG_TX_STAT[3] = sts_underflow (TX FIFO starved)

def tx_play(dma_buf):
    """Stream a CMA buffer of packed 32-bit samples out the MM2S/I2S TX path."""
    dma.write(REG_TX_ADDR_LO, dma_buf.device_address & 0xFFFFFFFF)
    dma.write(REG_TX_ADDR_HI, dma_buf.device_address >> 32)
    dma.write(REG_TX_BYTES,   dma_buf.nbytes)
    dma.write(REG_TX_START,   1)                       # cfg_start pulse

    # done (bit 31) is sticky from the previous run: wait for it to clear (the
    # reader took the new transfer) and then set again (this transfer finished).
    while dma.read(REG_TX_STAT) & DONE_BIT:            # stale done clears
        pass
    while not (dma.read(REG_TX_STAT) & DONE_BIT):      # new transfer done
        pass

    if dma.read(REG_TX_STAT) & UNDERFLOW_BIT:
        print("WARNING: playback underflow - reader could not keep the TX FIFO fed")
    return dma.read(REG_TX_BYTES_READ)

print(f"Playing {N/SAMPLE_RATE:.2f} s ({N} frames) out LINE OUT ...")
played = tx_play(buf)
print(f"Played {played} bytes ({played // 4} frames); playback complete!")

## Play in notebook
Since the samples are in 24-bit PCM format, 
users can play the audio directly in notebook.

In [ ]:
from IPython.display import Audio as IPAudio
IPAudio("capture.wav")

## Plotting PCM data

Users can display the audio data in notebook:

1. Plot the audio signal's amplitude over time.
2. Plot the spectrogram of the audio signal.

The next cell reads the saved audio file and processes it into a `numpy` array.
Note that if the audio sample width is not standard, additional processing
is required. In the following example, the `sample_width` is read from the
wave file itself (24-bit dual-channel PCM audio, where `sample_width` is 3 bytes).

In [ ]:
%matplotlib inline
import wave
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.fftpack import fft

wav_path = "capture.wav"
with wave.open(wav_path, 'r') as wav_file:
    raw_frames = wav_file.readframes(-1)
    num_frames = wav_file.getnframes()
    num_channels = wav_file.getnchannels()
    sample_rate = wav_file.getframerate()
    sample_width = wav_file.getsampwidth()
    
temp_buffer = np.empty((num_frames, num_channels, 4), dtype=np.uint8)
raw_bytes = np.frombuffer(raw_frames, dtype=np.uint8)
temp_buffer[:, :, :sample_width] = raw_bytes.reshape(-1, num_channels, 
                                                    sample_width)
temp_buffer[:, :, sample_width:] = \
    (temp_buffer[:, :, sample_width-1:sample_width] >> 7) * 255
frames = temp_buffer.view('<i4').reshape(temp_buffer.shape[:-1])

### 1. Amplitude over time

In [ ]:
for channel_index in range(num_channels):
    plt.figure(num=None, figsize=(15, 3))
    plt.title('Audio in Time Domain (Channel {})'.format(channel_index))
    plt.xlabel('Time in s')
    plt.ylabel('Amplitude')
    time_axis = np.arange(0, num_frames/sample_rate, 1/sample_rate)
    plt.plot(time_axis, frames[:, channel_index])
    plt.show()

### 2. Frequency spectrum

In [ ]:
for channel_index in range(num_channels):
    plt.figure(num=None, figsize=(15, 3))
    plt.title('Audio in Frequency Demain (Channel {})'.format(channel_index))
    plt.xlabel('Frequency in Hz')
    plt.ylabel('Magnitude')
    temp = fft(frames[:, channel_index])
    yf = temp[1:len(temp)//2]
    xf = np.linspace(0.0, sample_rate/2, len(yf))
    plt.plot(xf, abs(yf))
    plt.show()

### 3. Frequency spectrum over time
Use the `classic` plot style for better display.

In [ ]:
for channel_index in range(num_channels):
    np.seterr(divide='ignore', invalid='ignore')
    matplotlib.style.use("classic")
    plt.figure(num=None, figsize=(15, 3))
    plt.title('Signal Spectogram (Channel {})'.format(channel_index))
    plt.xlabel('Time in s')
    plt.ylabel('Frequency in Hz')
    plt.specgram(frames[:, channel_index], Fs=sample_rate)

Copyright (c) 2022 Xilinx, Inc 
<br>
Copyright (C) 2022-2025 Advanced Micro Devices, Inc. 
<br>
SPDX-License-Identifier: BSD-3-Clause 